# Retrieval-Augmented Generation (RAG) — v2

<sup>This notebook is a part of the Natural Language Processing class at the University of Ljubljana, Faculty of Computer and Information Science.</sup>

---

Large Language Models are remarkable at reasoning, but they have one fundamental limitation: **their knowledge is frozen at the time of training**. Once trained, the model cannot learn new facts unless it is retrained or fine-tuned — both expensive operations.

**Retrieval-Augmented Generation (RAG)** solves this by giving the model access to an external knowledge base *at inference time*. Instead of relying solely on memorised parameters, the model can look up relevant passages and ground its answer in real documents.

> **Analogy.** RAG is to an LLM what an **open-book exam** is to a student. The student (model) is not expected to memorise every fact; instead, they are allowed to consult textbooks (knowledge base) and reason from what they find.

---

## What you will learn

| Section | Topic |
|---|---|
| 1 | The hallucination problem: why pure LLMs fail on factual questions |
| 2 | Loading and inspecting a knowledge base (Wikipedia) |
| 3 | Text splitting strategies and why chunk size matters |
| 4 | Embeddings: turning text into vectors |
| 5 | Vector stores and FAISS indexing |
| 6 | Retrieval: similarity search vs. MMR |
| 7 | Building a RAG chain with modern **LCEL** syntax |
| 8 | Without RAG vs. with RAG — a direct comparison |
| 9 | Conversational RAG with memory |
| 10 | Exercises |

---

## RAG Architecture

The RAG pipeline has two phases:

**Indexing (done once, offline)**
```
Documents  →  Text Splitter  →  Embedding Model  →  Vector Store
```

**Retrieval + Generation (done at query time)**
```
User Query  →  Embedding Model  →  Vector Store (search)  →  Top-k Chunks
                                                                    ↓
                                          Prompt = [System + Context + Query]  →  LLM  →  Answer
```

The key insight is that **the query and documents are embedded into the same vector space**, so semantic similarity can be measured with cosine distance, regardless of exact word overlap.

## 1  Setup

We only need a handful of lightweight libraries. All model inference (the chat model **and** the embeddings) runs through the **OpenAI API**, so there is no `torch`, `transformers`, GPU, or quantisation to worry about.

| Library | Purpose |
|---|---|
| `langchain` / `langchain-community` / `langchain-core` | Document loaders, chains, retrievers |
| `langchain-openai` | LangChain <-> OpenAI bridge (chat model + embeddings) |
| `langchain-text-splitters` | Splitting documents into chunks |
| `faiss-cpu` | Fast vector similarity search |
| `wikipedia` | Wikipedia document loader |
| `tiktoken` | Token counting (used by the token splitter) |
| `python-dotenv` | Loads your `OPENAI_API_KEY` from a local `.env` file |

In [ ]:
# !pip install -q -U langchain langchain-community langchain-core langchain-text-splitters langchain-openai
# !pip install -q -U faiss-cpu wikipedia tiktoken python-dotenv

In [1]:
# Load your OpenAI API key from a local .env file (which must NOT be committed).
# The .env file should contain a single line:  OPENAI_API_KEY=sk-...
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found - add it to your .env file."
print("OpenAI API key loaded.")

OpenAI API key loaded.


In [2]:
import warnings
warnings.filterwarnings("ignore")

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, TokenTextSplitter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

print("Imports ready.")

Imports ready.


## 2  The Problem: LLMs Hallucinate Facts

Before adding any retrieval, let's demonstrate the core problem that RAG solves.

LLMs generate text by predicting the next token based on patterns in training data. When asked a factual question about something they don't know well, they don't say "I don't know" - they **hallucinate** plausible-sounding but wrong answers.

We will use OpenAI's **`gpt-4o-mini`** as our LLM. It is called over the API, so nothing is downloaded or run locally - you only need your `OPENAI_API_KEY`.

We set `temperature=0` so answers are (nearly) deterministic: the same prompt yields the same answer, which keeps the *without-RAG* vs. *with-RAG* comparison fair.

In [4]:
# A single ChatOpenAI object is our LLM for the whole notebook.
# Swap `model` for any chat model your key can access (e.g. "gpt-4o", "gpt-4.1-mini").
LLM_MODEL = "gpt-4o-mini"

llm = ChatOpenAI(model=LLM_MODEL, temperature=0, max_tokens=1024)

print(f"Chat model ready: {LLM_MODEL}")

Chat model ready: gpt-4o-mini


In [5]:
# Quick sanity check that the API key and model work.
print(llm.invoke("Reply with exactly: RAG notebook is ready.").content)

RAG notebook is ready.


In [6]:
# A plain (no-retrieval) chain: prompt -> LLM -> string.
# ChatOpenAI takes structured chat messages, so we build them with ChatPromptTemplate
# rather than a model-specific chat template.

plain_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer the question concisely and accurately."),
    ("human", "{question}"),
])

plain_chain = plain_prompt | llm | StrOutputParser()

In [7]:
# Ask a highly specific factual question — something the model may not know well
question = "What are the key differences between BERT and RoBERTa in terms of training procedure?"

print("=== WITHOUT RAG ===")
answer_no_rag = plain_chain.invoke({"question": question})
print(answer_no_rag)

=== WITHOUT RAG ===


BERT and RoBERTa have several key differences in their training procedures:

1. **Training Data**: 
   - **BERT** was trained on the BookCorpus and English Wikipedia.
   - **RoBERTa** used a larger dataset, including the same BookCorpus and English Wikipedia, but also added more data from sources like Common Crawl, resulting in a significantly larger training corpus.

2. **Training Objective**:
   - Both models use the masked language modeling (MLM) objective, but RoBERTa removes the Next Sentence Prediction (NSP) objective that BERT uses, focusing solely on MLM.

3. **Dynamic Masking**:
   - **BERT** uses static masking, where the same tokens are masked in every epoch.
   - **RoBERTa** employs dynamic masking, where the masked tokens change in each epoch, allowing for more diverse training.

4. **Batch Size and Training Steps**:
   - **RoBERTa** typically uses larger batch sizes and trains for more steps compared to BERT, which contributes to better performance.

5. **Hyperparameter T

In [8]:
# Ask a highly specific factual question — something the model may not know well
question = "In what year was the founding stallion Siglavy foaled and where did he originally come from?"

print("=== WITHOUT RAG ===")
answer_no_rag = plain_chain.invoke({"question": question})
print(answer_no_rag)

=== WITHOUT RAG ===


The founding stallion Siglavy was foaled in 1878 and originally came from the Arabian Peninsula, specifically from the region of Syria.


The model may give a reasonable-sounding answer, but it is reconstructing facts from memory — which can contain errors or omissions. Now let's build a RAG system that **retrieves the actual source text** before answering.

## 3  Building the Knowledge Base

### 3.1  Loading Documents

We use **Wikipedia** as our knowledge source. It has two advantages for demos:

1. No authentication or scraping needed — `WikipediaLoader` fetches articles via the Wikipedia API.
2. The content is well-structured and factual, making it easy to verify whether the model's answer is grounded.

We load several NLP-related articles to give our RAG system a rich knowledge base.

In [9]:
# Wikipedia's API now requires a descriptive User-Agent header, otherwise it
# returns HTTP 403 (see https://w.wiki/4wJS). Set one before loading.
import wikipedia
wikipedia.set_user_agent("DGT-Summer-School/1.0 (NLP course material; contact: ales.zagar@fri.uni-lj.si)")

# Articles to include in our knowledge base
topics = [
    "BERT (language model)",
    "GPT (language model)",
    "Transformer (deep learning architecture)",
    "Word2vec",
    "Attention mechanism",
    "Transfer learning",
    "Natural language processing",
    "Lipizzan",
    "Primož Trubar",
    "Idrija mercury mine",
]

print("Fetching Wikipedia articles...")
all_docs = []
for topic in topics:
    loader = WikipediaLoader(query=topic, load_max_docs=1, doc_content_chars_max=30000)
    docs = loader.load()
    all_docs.extend(docs)
    print(f"  Loaded: '{topic}' ({len(docs[0].page_content)} chars)")

print(f"\nTotal documents: {len(all_docs)}")
print(f"Total characters: {sum(len(d.page_content) for d in all_docs):,}")

Fetching Wikipedia articles...


  Loaded: 'BERT (language model)' (17537 chars)


  Loaded: 'GPT (language model)' (14543 chars)


  Loaded: 'Transformer (deep learning architecture)' (30000 chars)


  Loaded: 'Word2vec' (29636 chars)


...

  Loaded: 'Transfer learning' (8654 chars)


  Loaded: 'Natural language processing' (30000 chars)


  Loaded: 'Lipizzan' (22504 chars)


  Loaded: 'Primož Trubar' (7430 chars)


  Loaded: 'Idrija mercury mine' (6941 chars)

Total documents: 10
Total characters: 192,836


In [10]:
# Inspect one document
print("=== Document metadata ===")
print(all_docs[0].metadata)
print("\n=== First 500 characters ===")
print(all_docs[0].page_content[:500])

=== Document metadata ===
{'title': 'BERT (language model)', 'summary': 'Bidirectional encoder representations from transformers (BERT) is a language model introduced in October 2018 by researchers at Google. It learns to represent text as a sequence of vectors using self-supervised learning. It uses the encoder-only transformer architecture. BERT dramatically improved the state of the art for large language models. As of 2020, BERT is a ubiquitous baseline in natural language processing (NLP) experiments. \nBERT is trained by masked token prediction and next sentence prediction. With this training, BERT learns contextual, latent representations of tokens in their context, similar to ELMo and GPT-2. It found applications for many natural language processing tasks, such as coreference resolution and polysemy resolution. It improved on ELMo and spawned the study of "BERTology", which attempts to interpret what is learned by BERT.\nBERT was originally implemented in the English language a

### 3.2  Text Splitting

Embedding models and LLM context windows both have **token limits**. A full Wikipedia article may have 10,000+ characters — too long to embed as a single unit or include in a prompt wholesale.

We must split documents into **chunks**: shorter, self-contained passages that can be:
1. Embedded individually into vectors.
2. Passed selectively to the LLM based on relevance.

**Key parameters:**

| Parameter | Effect |
|---|---|
| `chunk_size` | Maximum size of each chunk (in characters or tokens) |
| `chunk_overlap` | How many characters from the end of one chunk are repeated at the start of the next |

**Why overlap?** Important context may straddle a chunk boundary. Overlap ensures that such information appears in at least one complete chunk. Too much overlap wastes space; too little risks missing cross-boundary context.

**Choosing chunk size:**
- **Too small** → chunks may lack enough context to answer a question; retrieval may be noisy.
- **Too large** → fewer, denser chunks; important passages are diluted by surrounding text; may overflow embedding model limits.
- A common starting point: **512–1024 characters** (roughly 100–200 tokens).

In [11]:
# RecursiveCharacterTextSplitter tries to split on natural boundaries first:
# paragraphs (\n\n) → sentences (\n) → words ( ) → characters
# This preserves semantic coherence better than a hard character cut.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,          # Target size in characters
    chunk_overlap=100,       # 100-char overlap between consecutive chunks
    length_function=len,
    add_start_index=True,    # Store the original char offset in metadata
)

chunks = text_splitter.split_documents(all_docs)

print(f"Documents → chunks: {len(all_docs)} → {len(chunks)}")
print(f"Average chunk size : {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars")
print(f"Min / Max          : {min(len(c.page_content) for c in chunks)} / {max(len(c.page_content) for c in chunks)} chars")

Documents → chunks: 10 → 380
Average chunk size : 519 chars
Min / Max          : 5 / 799 chars


### 3.3  Comparing Chunk Sizes

To build intuition, let's see how different `chunk_size` values affect the number and character of the chunks.

In [12]:
print(f"{'chunk_size':>12}  {'num_chunks':>10}  {'avg_len':>8}")
print("-" * 36)
for size in [200, 400, 800, 1600]:
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=size // 8)
    ch = sp.split_documents(all_docs)
    avg = sum(len(c.page_content) for c in ch) / len(ch)
    print(f"{size:>12}  {len(ch):>10}  {avg:>8.0f}")

  chunk_size  num_chunks   avg_len
------------------------------------
         200        1353       146
         400         739       271
         800         380       519
        1600         180      1092


## 4  Embeddings

An **embedding model** maps text to a fixed-size dense vector. Texts with similar meaning end up close together in this high-dimensional space, enabling semantic (meaning-based) search rather than keyword matching.

```
"BERT uses masked language modelling"  ->  [0.12, -0.87, 0.33, ...]
"BERT is trained with MLM"             ->  [0.14, -0.85, 0.31, ...]   <- close!
"The weather is sunny today"           ->  [-0.9,  0.23, 0.71, ...]   <- far away
```

We use OpenAI's **`text-embedding-3-small`** - a fast, inexpensive model that returns 1536-dimensional vectors. OpenAI embeddings are already L2-normalised, so the dot product of two vectors equals their cosine similarity.

In [13]:
EMBED_MODEL = "text-embedding-3-small"

embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

print(f"Embedding model ready: {EMBED_MODEL}")

Embedding model ready: text-embedding-3-small


In [14]:
# Demonstrate semantic similarity directly on embeddings
import numpy as np

sentences = [
    "BERT uses a masked language model pre-training objective.",
    "BERT is trained by masking tokens and predicting them.",
    "The Eiffel Tower is located in Paris, France.",
]

vecs = embeddings.embed_documents(sentences)
vecs = np.array(vecs)  # shape: (3, 1536)

# Cosine similarity matrix (OpenAI vectors are L2-normalised, so dot product = cosine)
sim = vecs @ vecs.T

print("Cosine similarity matrix:")
print(f"{'':50s}  S1    S2    S3")
labels = ["S1: BERT MLM objective", "S2: BERT masking tokens", "S3: Eiffel Tower"]
for i, label in enumerate(labels):
    row = "  ".join(f"{sim[i,j]:.3f}" for j in range(3))
    print(f"{label:50s}  {row}")

print("\nS1 and S2 are semantically similar -> high cosine similarity.")
print("S3 is about a different topic -> low similarity with S1/S2.")

Cosine similarity matrix:
                                                    S1    S2    S3
S1: BERT MLM objective                              1.000  0.740  0.019
S2: BERT masking tokens                             0.740  1.001  0.002
S3: Eiffel Tower                                    0.019  0.002  0.999

S1 and S2 are semantically similar -> high cosine similarity.
S3 is about a different topic -> low similarity with S1/S2.


## 5  Vector Store (FAISS)

A **vector store** indexes all embedded chunks so we can find the nearest neighbours of a query vector efficiently.

**FAISS** (Facebook AI Similarity Search) is a library for fast approximate nearest-neighbour search in high-dimensional spaces. For our use case:

1. Each chunk is embedded into a 384-dimensional vector.
2. All vectors are stored in FAISS.
3. At query time: embed the query → search FAISS → return the top-k most similar chunk vectors.

This is **much faster** than computing cosine similarity against every stored vector sequentially.

[FAISS and langchain docs](https://docs.langchain.com/oss/python/integrations/vectorstores/faiss)

In [15]:
print(f"Indexing {len(chunks)} chunks into FAISS...")
vectorstore = FAISS.from_documents(chunks, embeddings)
print("Done.")
print(f"Index size: {vectorstore.index.ntotal} vectors of dimension {vectorstore.index.d}")

Indexing 380 chunks into FAISS...


Done.
Index size: 380 vectors of dimension 1536


## 6  Retrieval: Similarity Search vs. MMR

The retriever is the component that takes a query and returns the most relevant chunks.

### 6.1  Similarity Search

Plain **similarity search** returns the top-k chunks with the highest cosine similarity to the query. Simple and effective, but it can return **redundant** results: if the top-5 chunks all discuss the same narrow sub-topic, important tangential context is missed.

### 6.2  Maximum Marginal Relevance (MMR)

**MMR** balances relevance and diversity. It selects chunks that are:
- **Relevant** to the query (high similarity to query)
- **Diverse** from already-selected chunks (low similarity to each other)

The trade-off is controlled by `lambda_mult` (0 = max diversity, 1 = max relevance).

MMR is better when you want broad coverage of a topic rather than several chunks that all say the same thing.

In [16]:
query = "How does BERT differ from GPT in terms of training objectives?"

# --- Similarity search ---
sim_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)
sim_results = sim_retriever.invoke(query)

print("=== Similarity Search Results ===")
for i, doc in enumerate(sim_results):
    title = doc.metadata.get("title", "unknown")
    print(f"[{i+1}] Source: {title}")
    print(doc.page_content[:200])
    print()

=== Similarity Search Results ===
[1] Source: BERT (language model)
BERT is trained by masked token prediction and next sentence prediction. With this training, BERT learns contextual, latent representations of tokens in their context, similar to ELMo and GPT-2. It fo

[2] Source: BERT (language model)
BERT was originally published by Google researchers Jacob Devlin, Ming-Wei Chang, Kenton Lee, and Kristina Toutanova. The design has its origins from pre-training contextual representations, including

[3] Source: BERT (language model)
=== Fine-tuning ===

BERT is meant as a general pretrained model for various applications in natural language processing. That is, after pre-training, BERT can be fine-tuned with fewer resources on sm

[4] Source: BERT (language model)
== Interpretation ==
Language models like ELMo, GPT-2, and BERT, spawned the study of "BERTology", which attempts to interpret what is learned by these models. Their performance on these natural langu



In [17]:
# --- MMR search ---
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.5},
    # fetch_k: initial candidate pool size (larger = better diversity quality)
    # lambda_mult: relevance weight (0.5 = equal balance)
)
mmr_results = mmr_retriever.invoke(query)

print("=== MMR Search Results ===")
for i, doc in enumerate(mmr_results):
    title = doc.metadata.get("title", "unknown")
    print(f"[{i+1}] Source: {title}")
    print(doc.page_content[:200])
    print()

=== MMR Search Results ===
[1] Source: BERT (language model)
BERT is trained by masked token prediction and next sentence prediction. With this training, BERT learns contextual, latent representations of tokens in their context, similar to ELMo and GPT-2. It fo

[2] Source: Generative pre-trained transformer
This phenomenon has influenced the development of larger GPT models and contributed to their increased effectiveness across a wide range of tasks.

[3] Source: Generative pre-trained transformer
On June 11, 2018, OpenAI researchers and engineers published a paper called "Improving Language Understanding by Generative Pre-Training", which introduced GPT-1, the first GPT model. It was designed 

[4] Source: BERT (language model)
== Architecture ==

BERT is an "encoder-only" transformer architecture. At a high level, BERT consists of 4 modules:



In [18]:
# Count how many distinct source articles each retrieval strategy returns
sim_sources = {d.metadata.get("title", "?") for d in sim_results}
mmr_sources = {d.metadata.get("title", "?") for d in mmr_results}

print(f"Similarity search: {len(sim_sources)} distinct sources → {sim_sources}")
print(f"MMR search       : {len(mmr_sources)} distinct sources → {mmr_sources}")
print("\nMMR typically returns results from more diverse sources.")

Similarity search: 1 distinct sources → {'BERT (language model)'}
MMR search       : 2 distinct sources → {'BERT (language model)', 'Generative pre-trained transformer'}

MMR typically returns results from more diverse sources.


## 7  Building the RAG Chain with LCEL

**LangChain Expression Language (LCEL)** is LangChain's modern way of composing chains using the pipe operator `|`. Each component receives output from the previous step:

```python
chain = step_1 | step_2 | step_3
chain.invoke(input)  # passes input through all steps
```

This replaces the older `ConversationalRetrievalChain` and similar high-level abstractions, which are now deprecated. LCEL is more transparent, composable, and supports streaming out of the box.

### Full RAG chain

```
Question
   ├──────────────────────────────────► RunnablePassthrough (keeps question)
   │                                             │
   └──► Retriever ──► format_docs (join)         │
                           │                     │
                    {context: ..., question: ...} │
                                    ↓
                               PromptTemplate
                                    ↓
                                   LLM
                                    ↓
                              StrOutputParser
```

In [19]:
# Choose our retriever (MMR for diversity)
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.6},
)

# Helper to concatenate retrieved Documents into a single context string
def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[Source: {d.metadata.get('title', 'unknown')}]\n{d.page_content}"
        for d in docs
    )

In [20]:
# RAG prompt as a ChatPromptTemplate. It receives a dict {context, question}.

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     """You are a precise, helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't have enough information to answer that."
Do not make up facts. Cite which source(s) you used at the end of your answer.

CONTEXT:
{context}"""),
    ("human", "{question}"),
])

In [21]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

In [22]:
# Helper to print the answer neatly
def ask(question: str):
    print(f"Question: {question}")
    print("-" * 60)
    answer = rag_chain.invoke(question)
    print(answer)
    print("=" * 60)
    return answer

In [23]:
ask("What are the key differences between BERT and GPT in terms of their architecture and training objectives?")

Question: What are the key differences between BERT and GPT in terms of their architecture and training objectives?
------------------------------------------------------------


I don't have enough information to answer that.


"I don't have enough information to answer that."

In [24]:
ask("What is the attention mechanism and why was it introduced?")

Question: What is the attention mechanism and why was it introduced?
------------------------------------------------------------


The attention mechanism is a technique developed to address the weaknesses of using information from the hidden layers of recurrent neural networks (RNNs). RNNs tend to favor information contained in words at the end of a sentence, which can lead to the attenuation of significance and predictive weight assigned to information earlier in the sentence. The attention mechanism allows a token to have equal access to any part of a sentence directly, rather than only through the previous state. This enables better context understanding and captures global dependencies in the input data.

(Source: Attention (machine learning))


'The attention mechanism is a technique developed to address the weaknesses of using information from the hidden layers of recurrent neural networks (RNNs). RNNs tend to favor information contained in words at the end of a sentence, which can lead to the attenuation of significance and predictive weight assigned to information earlier in the sentence. The attention mechanism allows a token to have equal access to any part of a sentence directly, rather than only through the previous state. This enables better context understanding and captures global dependencies in the input data.\n\n(Source: Attention (machine learning))'

In [25]:
ask("How does Word2vec represent words as vectors?")

Question: How does Word2vec represent words as vectors?
------------------------------------------------------------


Word2vec represents words as high-dimension vectors of numbers that capture relationships between words. It maps words that appear in similar contexts to vectors that are nearby in terms of cosine similarity, indicating the level of semantic similarity between the words. Each unique word in a large corpus of text is assigned a vector in a vector space, typically of several hundred dimensions. This is achieved through shallow, two-layer neural networks that are trained to reconstruct linguistic contexts of words. 

[Source: Word2vec]


'Word2vec represents words as high-dimension vectors of numbers that capture relationships between words. It maps words that appear in similar contexts to vectors that are nearby in terms of cosine similarity, indicating the level of semantic similarity between the words. Each unique word in a large corpus of text is assigned a vector in a vector space, typically of several hundred dimensions. This is achieved through shallow, two-layer neural networks that are trained to reconstruct linguistic contexts of words. \n\n[Source: Word2vec]'

In [26]:
ask("In what year was the founding stallion Siglavy foaled and where did he originally come from?")

Question: In what year was the founding stallion Siglavy foaled and where did he originally come from?
------------------------------------------------------------


Siglavy was foaled in 1810 and originally came from Syria. [Source: Lipizzan]


'Siglavy was foaled in 1810 and originally came from Syria. [Source: Lipizzan]'

## 8  Without RAG vs. With RAG — Side-by-Side

This is the core experiment. We ask the same specific factual question to:
1. The plain LLM (no context)
2. The RAG-augmented LLM (with retrieved Wikipedia context)

After running both cells, compare:
- **Specificity**: Does the RAG answer include concrete details absent from the baseline?
- **Grounding**: Does the RAG answer cite sources and stick to them?
- **Accuracy**: Is either answer factually wrong?

In [27]:
comparison_question = "In what year was the founding stallion Siglavy foaled and where did he originally come from?"

print("=" * 60)
print("WITHOUT RAG (pure LLM memory)")
print("=" * 60)
baseline = plain_chain.invoke({"question": comparison_question})
print(baseline)

WITHOUT RAG (pure LLM memory)


The founding stallion Siglavy was foaled in 1878 and originally came from the Arabian Peninsula, specifically from the region of Syria.


In [28]:
print("=" * 60)
print("WITH RAG (retrieved Wikipedia context)")
print("=" * 60)
rag_answer = rag_chain.invoke(comparison_question)
print(rag_answer)

WITH RAG (retrieved Wikipedia context)


Siglavy was foaled in 1810 and originally came from Syria. [Source: Lipizzan]


In [29]:
# Also show what context was actually retrieved
retrieved = retriever.invoke(comparison_question)
print("=" * 60)
print(f"Retrieved {len(retrieved)} chunks:")
print("=" * 60)
for i, doc in enumerate(retrieved):
    print(f"\n[Chunk {i+1}] Source: {doc.metadata.get('title', 'unknown')}")
    print(doc.page_content[:1000], "...")

Retrieved 5 chunks:

[Chunk 1] Source: Lipizzan
Pluto: a gray Spanish stallion from the Royal Danish Stud, foaled in 1765
Conversano: a black Neapolitan stallion, foaled in 1767
Maestoso: a gray stallion from the Kladrub stud with a Spanish dam, foaled 1773, descendants today all trace via Maestoso X, foaled in Hungary in 1819
Favory: a dun stallion from the Kladrub stud, foaled in 1779
Neapolitano: a bay Neapolitan stallion from the Polesine, foaled in 1790
Siglavy: a gray Arabian stallion, originally from Syria, foaled in 1810
Two additional stallion lines are found in Croatia, Hungary, and other eastern European countries, as well as in North America. They are accepted as equal to the six classical lines by the Lipizzan International Federation. These are: ...

[Chunk 2] Source: Lipizzan
The principles taught at the Spanish Riding School are based on practices taught to cavalry riders to prepare their horses for warfare.  Young stallions come to the Spanish Riding School for trainin

## 9  Conversational RAG

So far our RAG chain handles single-turn questions. But real assistants need to maintain conversation history — a follow-up question like "Tell me more about that" only makes sense if the model remembers what "that" referred to.

**The challenge:** If the user says "How was it trained?" in a follow-up, the retriever receives this ambiguous query and may retrieve irrelevant chunks. We need to **reformulate** the question using the chat history before retrieval.

The modern LangChain approach uses two chains:
1. **History-aware retriever**: rewrites the follow-up question into a standalone question using chat history.
2. **Question-answering chain**: answers the standalone question using retrieved context.

```
Chat History + Follow-up Question
          ↓
  Reformulation Prompt + LLM
          ↓
   Standalone Question
          ↓
      Retriever
          ↓
  RAG Answer Chain
```

In [30]:
# Step 1: Rewrite a follow-up question into a standalone question using chat history.

contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given the conversation so far and a follow-up question, reformulate the follow-up "
     "as a standalone question that can be understood without the history. "
     "Return ONLY the reformulated question, nothing else."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

contextualize_chain = contextualize_prompt | llm | StrOutputParser()

def contextualized_question(input_dict: dict) -> str:
    """If there is chat history, rewrite the question; otherwise pass it through."""
    if input_dict.get("chat_history"):
        return contextualize_chain.invoke(input_dict)
    return input_dict["input"]

In [31]:
# QA prompt for conversational RAG.

conv_rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are a helpful NLP tutor. Answer the question using only the context below.
If the answer is not in the context, say you don't know. Be concise.

CONTEXT:
{context}"""),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

conversational_rag_chain = (
    RunnablePassthrough.assign(
        context=RunnableLambda(contextualized_question) | retriever | format_docs
    )
    | conv_rag_prompt
    | llm
    | StrOutputParser()
)

In [32]:
# Stateful conversation loop
chat_history = []

def chat_with_rag(user_message: str) -> str:
    # The chain now returns a string directly (pure LCEL, no wrapper dict)
    answer = conversational_rag_chain.invoke({
        "input": user_message,
        "chat_history": chat_history,
    })
    # Append the exchange to history for next turn
    chat_history.append(HumanMessage(content=user_message))
    chat_history.append(AIMessage(content=answer))
    return answer

In [33]:
# Turn 1: Initial question
q1 = "What is the Transformer architecture?"
print(f"User: {q1}")
a1 = chat_with_rag(q1)
print(f"Assistant: {a1}")

User: What is the Transformer architecture?


Assistant: The Transformer architecture is a deep learning model designed for natural language processing (NLP) that uses an attention mechanism to process entire sequences of text at once. It was introduced in the paper "Attention Is All You Need" in 2017 and solved many performance issues associated with older recurrent neural network (RNN) designs. The architecture allows for the training of larger and more sophisticated models capable of processing, mining, organizing, connecting, contrasting, and summarizing texts, as well as answering questions from textual input.


In [34]:
# Turn 2: Follow-up that references the previous answer
# Note: "it" refers to the Transformer — the system must use history to resolve this
q2 = "What problem was it designed to solve?"
print(f"User: {q2}")
a2 = chat_with_rag(q2)
print(f"Assistant: {a2}")

User: What problem was it designed to solve?


Assistant: The Transformer architecture was designed to solve performance issues associated with older recurrent neural network (RNN) designs for natural language processing (NLP).


In [35]:
# Turn 3: Another follow-up
q3 = "How does the self-attention mechanism achieve this?"
print(f"User: {q3}")
a3 = chat_with_rag(q3)
print(f"Assistant: {a3}")

User: How does the self-attention mechanism achieve this?


Assistant: The self-attention mechanism achieves this by allowing each element in the input sequence to attend to all other elements, enabling the model to capture global dependencies. This replaces the need for recurrence, allowing for more efficient processing of sequences and improving the model's ability to understand context.


In [36]:
# Print the full conversation history
print("=== Full Conversation History ===")
for msg in chat_history:
    role = "User" if isinstance(msg, HumanMessage) else "Assistant"
    print(f"\n[{role}]: {msg.content[:300]}")

=== Full Conversation History ===

[User]: What is the Transformer architecture?

[Assistant]: The Transformer architecture is a deep learning model designed for natural language processing (NLP) that uses an attention mechanism to process entire sequences of text at once. It was introduced in the paper "Attention Is All You Need" in 2017 and solved many performance issues associated with old

[User]: What problem was it designed to solve?

[Assistant]: The Transformer architecture was designed to solve performance issues associated with older recurrent neural network (RNN) designs for natural language processing (NLP).

[User]: How does the self-attention mechanism achieve this?

[Assistant]: The self-attention mechanism achieves this by allowing each element in the input sequence to attend to all other elements, enabling the model to capture global dependencies. This replaces the need for recurrence, allowing for more efficient processing of sequences and improving the model's abilit

## 10  Inspecting What Gets Retrieved

Understanding *why* a RAG system gives a particular answer requires looking at the retrieved chunks. A retrieval-augmented answer is only as good as the chunks it receives — garbage in, garbage out.

Good diagnostics:
- Are the retrieved chunks from the correct source?
- Are they relevant to the question?
- Is there redundancy (same passage retrieved twice)?

In [37]:
def rag_with_sources(question: str):
    """Run RAG and print both the answer and the source chunks used."""
    retrieved = retriever.invoke(question)
    context = format_docs(retrieved)
    
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke({
        "context": context,
        "question": question,
    })
    
    print(f"Question: {question}")
    print("-" * 60)
    print("ANSWER:")
    print(answer)
    print()
    print("RETRIEVED CONTEXT:")
    for i, doc in enumerate(retrieved):
        title = doc.metadata.get("title", "unknown")
        print(f"  [{i+1}] {title}: {doc.page_content[:150]}...")
    print("=" * 60)

rag_with_sources("What is transfer learning and how is it used in NLP?")

Question: What is transfer learning and how is it used in NLP?
------------------------------------------------------------
ANSWER:
Transfer learning (TL) is a technique in machine learning where knowledge learned from one task is reused to improve performance on a related task. In the context of natural language processing (NLP), transfer learning can enhance tasks such as text classification, natural language understanding, and natural language generation by applying knowledge gained from previous tasks to new ones. This approach can significantly improve learning efficiency by leveraging existing information.

[Source: Transfer learning, Natural language processing]

RETRIEVED CONTEXT:
  [1] Transfer learning: Transfer learning (TL) is a technique in machine learning (ML) in which knowledge learned from a task is re-used in order to boost performance on a re...
  [2] Natural language processing: Natural language processing (NLP) is the processing of natural language information by a

In [38]:
# Test with a question outside our knowledge base — the model should admit ignorance
rag_with_sources("What is the population of Buenos Aires?")

Question: What is the population of Buenos Aires?
------------------------------------------------------------
ANSWER:
I don't have enough information to answer that.

RETRIEVED CONTEXT:
  [1] BERT (language model): BERT was originally implemented in the English language at two model sizes, BERTBASE (110 million parameters) and BERTLARGE (340 million parameters). ...
  [2] Lipizzan: The Lipizzan breed suffered a setback to its population when a viral epidemic hit the Piber Stud in 1983. Forty horses and 8% of the expected foal cro...
  [3] Primož Trubar: 1565. While in Ljubljana, he lived in a house, on today's Fish Square (Ribji trg), in the oldest part of the city. Living in Ljubljana had profound im...
  [4] BERT (language model): The three attention matrices are added together element-wise, then passed through a softmax layer and multiplied by a projection matrix.
Absolute posi...
  [5] Lipizzan: === Spanish Riding School ===

The Spanish Riding School uses highly trained Lipizzan 

## 11  Chunking Strategy Comparison

The choice of text splitter affects retrieval quality. Let's compare:

| Splitter | How it splits | Best for |
|---|---|---|
| `RecursiveCharacterTextSplitter` | Tries paragraph → sentence → char boundaries | General prose |
| `TokenTextSplitter` | Splits by token count (using a tokenizer) | Embedding models with token limits |

The `TokenTextSplitter` is more precise for models that have strict token-count limits, since character length and token count don't map 1:1 (a character with an accent may tokenise to 2 tokens).

In [39]:
# Token-based splitting
token_splitter = TokenTextSplitter(
    chunk_size=200,     # 200 tokens per chunk (~150 words)
    chunk_overlap=20,
)
token_chunks = token_splitter.split_documents(all_docs)

print(f"RecursiveCharacter (800 chars) : {len(chunks)} chunks")
print(f"Token (200 tokens)             : {len(token_chunks)} chunks")

# Show first chunk from each
print("\n--- RecursiveCharacter chunk ---")
print(chunks[0].page_content[:300])
print("\n--- Token chunk ---")
print(token_chunks[0].page_content[:300])

RecursiveCharacter (800 chars) : 380 chunks
Token (200 tokens)             : 382 chunks

--- RecursiveCharacter chunk ---
Bidirectional encoder representations from transformers (BERT) is a language model introduced in October 2018 by researchers at Google. It learns to represent text as a sequence of vectors using self-supervised learning. It uses the encoder-only transformer architecture. BERT dramatically improved t

--- Token chunk ---
Bidirectional encoder representations from transformers (BERT) is a language model introduced in October 2018 by researchers at Google. It learns to represent text as a sequence of vectors using self-supervised learning. It uses the encoder-only transformer architecture. BERT dramatically improved t


In [40]:
# Build a second vectorstore with token-based chunks and compare retrieval
vectorstore_token = FAISS.from_documents(token_chunks, embeddings)
token_retriever = vectorstore_token.as_retriever(search_kwargs={"k": 4})

test_query = "How does BERT pre-training work?"

char_results = retriever.invoke(test_query)
tok_results  = token_retriever.invoke(test_query)

print("=== Character-based chunks ===")
for d in char_results:
    print(f"  {d.metadata.get('title','?')} | {len(d.page_content)} chars")

print("\n=== Token-based chunks ===")
for d in tok_results:
    print(f"  {d.metadata.get('title','?')} | {len(d.page_content)} chars")

=== Character-based chunks ===
  BERT (language model) | 796 chars
  BERT (language model) | 677 chars
  BERT (language model) | 507 chars
  BERT (language model) | 572 chars
  BERT (language model) | 116 chars

=== Token-based chunks ===
  BERT (language model) | 957 chars
  BERT (language model) | 909 chars
  BERT (language model) | 976 chars
  BERT (language model) | 1026 chars


## 12  Summary

You have built a complete RAG system. Here is what each component does:

```
+-----------------------------------------------------------------+
|                         INDEXING PHASE                           |
|                                                                 |
|  Documents  ->  Text Splitter  ->  Embeddings  ->  Vector Store |
|  (Wikipedia)   (chunk_size=800) (text-embedding  (FAISS)        |
|                                   -3-small)                     |
+-----------------------------------------------------------------+

+-----------------------------------------------------------------+
|                        QUERY PHASE (LCEL)                        |
|                                                                 |
|  Question --> Retriever (MMR, k=5)                              |
|       |              |                                          |
|       |        Top-k Chunks                                     |
|       |              |                                          |
|       +------> Prompt Template --> LLM --> StrOutputParser      |
|                {context, question}   (gpt-4o-mini via API)      |
+-----------------------------------------------------------------+
```

**Key design decisions and their effects:**

| Decision | Options | Impact |
|---|---|---|
| Chunk size | 200-2000 chars | Smaller = more precise; larger = more context per chunk |
| Overlap | 0-20% of chunk size | Reduces boundary artefacts |
| Embedding model | `text-embedding-3-small` -> `text-embedding-3-large` | Speed/cost vs. quality |
| Retrieval type | similarity vs. MMR | Precision vs. diversity |
| k (num chunks) | 3-10 | More context = better answers, but higher latency and risk of dilution |
| LLM | `gpt-4o-mini` -> `gpt-4o` / `gpt-4.1` | Answer quality vs. cost |